# Day 10 · Exercise 4: Registry Round-Trip

**What you'll build:** `registry_round_trip(templates: list[dict]) -> list[dict]` — a function that builds a versioned prompt registry from a list of template dicts, saves it to a JSON file, reloads it into a fresh dict, and returns the reloaded entries as a list.

**Why it matters:** Saving and reloading a registry is the persistence layer that turns a throw-away in-memory dict into an auditable, version-controlled prompt store — the same pattern used in every production prompt management system.

## Your Implementation

In [ ]:
import json
import tempfile
from pathlib import Path
from string import Template


def registry_round_trip(templates: list[dict]) -> list[dict]:
    """Build a versioned registry, persist it to JSON, reload it, and return the entries.

    Each dict in `templates` must have:
        - 'key':          str  — registry key in 'name_vN' format (e.g. 'summarize_v1')
        - 'template':     str  — raw template text with $placeholder syntax
        - 'placeholders': list[str] — names of every placeholder the template expects

    Steps:
        1. Register every template entry into an internal registry dict.
           Raise ValueError if a key already exists (immutability rule).
        2. Save the registry to a temporary JSON file (use tempfile.mkstemp or
           a fixed path like '/tmp/registry_rt.json').
        3. Load the JSON file into a *fresh* dict (do not reuse the original).
        4. Return the reloaded registry values as a list of dicts, each containing
           the 'key', 'template', and 'placeholders' fields.

    Args:
        templates: List of dicts, each with 'key', 'template', and 'placeholders'.

    Returns:
        List of dicts loaded from the JSON file.  Each dict has the keys
        'key', 'template', and 'placeholders'.  The list is sorted by 'key'.

    Raises:
        ValueError: If any key appears more than once in `templates`.

    Example:
        templates = [
            {
                'key': 'translate_v1',
                'template': 'Translate the following to $language:\n\n$text',
                'placeholders': ['text', 'language'],
            },
            {
                'key': 'translate_v2',
                'template': 'Translate the text below into $language. Be literal.\n\n$text',
                'placeholders': ['text', 'language'],
            },
        ]
        result = registry_round_trip(templates)
        # result is a list of 2 dicts loaded from disk, sorted by 'key'
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

_SAMPLE_TEMPLATES = [
    {
        'key': 'translate_v1',
        'template': 'Translate the following to $language:\n\n$text',
        'placeholders': ['text', 'language'],
    },
    {
        'key': 'translate_v2',
        'template': 'Translate the text below into $language. Be literal.\n\n$text',
        'placeholders': ['text', 'language'],
    },
    {
        'key': 'classify_v1',
        'template': 'Classify the text as one of: $labels.\nReply with the label only.\n\n$text',
        'placeholders': ['text', 'labels'],
    },
]


def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and is callable
    try:
        assert callable(registry_round_trip), 'registry_round_trip is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a list with the correct number of entries
    try:
        result = registry_round_trip(_SAMPLE_TEMPLATES)
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        assert len(result) == 3, f'expected 3 entries, got {len(result)}'
        print(f'{_PASS} Check 2/{total}: returns a list with 3 entries (one per template)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: each entry contains 'key', 'template', and 'placeholders'
    try:
        result = registry_round_trip(_SAMPLE_TEMPLATES)
        for entry in result:
            assert 'key' in entry, f"entry missing 'key': {entry}"
            assert 'template' in entry, f"entry missing 'template': {entry}"
            assert 'placeholders' in entry, f"entry missing 'placeholders': {entry}"
        keys_returned = sorted(e['key'] for e in result)
        keys_expected = sorted(t['key'] for t in _SAMPLE_TEMPLATES)
        assert keys_returned == keys_expected, f'keys mismatch: {keys_returned} vs {keys_expected}'
        print(f'{_PASS} Check 3/{total}: each entry has key/template/placeholders and all keys round-tripped')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: duplicate key raises ValueError (immutability rule)
    try:
        duplicate_templates = [
            {
                'key': 'summarize_v1',
                'template': 'Summarize: $text',
                'placeholders': ['text'],
            },
            {
                'key': 'summarize_v1',
                'template': 'A different summarize prompt: $text',
                'placeholders': ['text'],
            },
        ]
        raised = False
        try:
            registry_round_trip(duplicate_templates)
        except ValueError:
            raised = True
        assert raised, 'expected ValueError for duplicate key, but none was raised'
        print(f'{_PASS} Check 4/{total}: duplicate key correctly raises ValueError')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: placeholder lists survive the JSON round-trip intact
    try:
        result = registry_round_trip(_SAMPLE_TEMPLATES)
        entry_map = {e['key']: e for e in result}
        v1 = entry_map.get('translate_v1')
        assert v1 is not None, "'translate_v1' missing from reloaded registry"
        assert sorted(v1['placeholders']) == ['language', 'text'], (
            f"placeholder list mismatch: {v1['placeholders']}"
        )
        print(f'{_PASS} Check 5/{total}: placeholder lists survive the JSON round-trip')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
    print(f'  {score}/{total} passed.' + ('' if score == total else ' Keep going!'))


_run_checks()

## Bonus Challenge

Extend `registry_round_trip` (or write a helper) to accept an existing JSON file path and *merge* the loaded registry with the new templates — skipping any key already present on disk rather than raising. This is the pattern Day 20 uses when a long-running service appends new prompt versions to a shared registry file without restarting.

```python
def merge_into_registry(path: str, new_templates: list[dict]) -> dict:
    """Load existing registry from path, add only new keys, save back."""
    ...
```

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json
import tempfile
from pathlib import Path


def registry_round_trip(templates: list[dict]) -> list[dict]:
    # Step 1: build an in-memory registry (enforce immutability)
    registry: dict[str, dict] = {}
    for t in templates:
        key = t['key']
        if key in registry:
            raise ValueError(
                f"Template '{key}' already exists. "
                "Create a new version instead of mutating an existing one."
            )
        registry[key] = {
            'template': t['template'],
            'placeholders': t['placeholders'],
        }

    # Step 2: save to a temporary JSON file
    tmp_path = Path(tempfile.mktemp(suffix='.json'))
    with open(tmp_path, 'w') as fh:
        json.dump(registry, fh, indent=2)

    # Step 3: reload into a fresh dict
    with open(tmp_path) as fh:
        reloaded: dict[str, dict] = json.load(fh)

    # Step 4: return as a sorted list of dicts with 'key' included
    return sorted(
        [{'key': k, **v} for k, v in reloaded.items()],
        key=lambda e: e['key'],
    )
```

**Why this works:** The registry is a plain Python dict whose keys encode both the template name and its version, so two versions of the same template coexist without conflict. Raising `ValueError` on a duplicate key enforces immutability in code rather than policy — it is impossible to accidentally overwrite an existing version. Because the registry contains only nested dicts, lists, and strings, `json.dump` serialises it without any custom serialiser, and `json.load` reconstructs it faithfully, proving that the round-trip preserves the full template text and placeholder list.
</details>